In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-pro",api_key=os.environ.get("GOOGLE_API_KEY"))

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001",google_api_key=os.environ.get("GOOGLE_API_KEY"))

In [4]:
# Helper function to format and print document content
def pretty_print_docs(docs):
    # Print each document in the list with a separator between them
    print(
        f"\n{'-' * 100}\n".join(  # Separator line for better readability
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]  # Format: Document number + content
        )
    )

In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    #text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    text_splitter = SemanticChunker(embeddings, breakpoint_threshold_type="gradient",breakpoint_threshold_amount=60.0)
    all_splits = text_splitter.split_documents(docs)
    return all_splits

documents=load_pdf("Ashish_Jain_AI_Engineer.pdf")




# Create FAISS index from documents and set up retriever
retriever = FAISS.from_documents(documents, embeddings).as_retriever(
    search_kwargs={"k": 10}
)

# Define the query
query = "what are the skills present in the resume ?"

# Execute the query and retrieve results
docs = retriever.invoke(query)

# Display the retrieved documents
pretty_print_docs(docs)

Document 1:

• Evaluated model performance using BLEU score and achieved significant improvements through iterative refinement. Technical Skills
Software Development: Python, OOPS, Django, Flask, FastAPI, SQL, C++, HTML, CSS, Gradio, Streamlit. Artificial Intelligence: Generative AI, Natural Language Processing, Computer Vision, Machine Learning, Deep Learning. Tools and Packages: Hugging Face, LangChain, Ollama, NumPy, Pandas, Scikit-learn, nltk, Pytorch, Keras, Tensorflow. Cloud Services : Azure, AWS
Achievements
• Completed 3 technical internships at small startups, gaining hands-on experience in software development and
problem-solving.
----------------------------------------------------------------------------------------------------
Document 2:

Built GitLab CI/CD pipelines for automated policy updates, storing configurations in Azure
Storage Account. • Developed OCR pipeline addressing data extraction challenges, automating conversion processes.
--------------------------------

In [9]:
vs.similarity_search("what are the skills present in the resume ?")

[Document(id='f4e56956-9712-4ad6-a9bb-ec25f5cb8592', metadata={'source': 'Ashish_Jain_AI_Engineer.pdf', 'page': 0, 'page_label': '1'}, page_content='Software Development: Python, OOPS, Django, Flask, FastAPI, SQL, C++, HTML, CSS, Gradio, Streamlit.\nArtificial Intelligence: Generative AI, Natural Language Processing, Computer Vision, Machine Learning, Deep Learning.\nTools and Packages: Hugging Face, LangChain, Ollama, NumPy, Pandas, Scikit-learn, nltk, Pytorch, Keras, Tensorflow.\nCloud Services : Azure, AWS\nAchievements\n• Completed 3 technical internships at small startups, gaining hands-on experience in software development and'),
 Document(id='0d60dd3f-caa0-489b-a997-1ba8de5af0ee', metadata={'source': 'Ashish_Jain_AI_Engineer.pdf', 'page': 0, 'page_label': '1'}, page_content='• Trained a sequence-to-sequence Recurrent Neural Network (RNN) architecture for English to German translation using\nkeras-NLP\n• Preprocessed and tokenized English and German corpora to prepare the data fo

In [16]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Initialize the model
model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Select the top 3 documents
compressor = CrossEncoderReranker(model=model, top_n=3)

# Initialize the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)



In [17]:
# Retrieve compressed documents
compressed_docs = compression_retriever.invoke("what are the skills present in the resume?")

# Display the documents
pretty_print_docs(compressed_docs)

Document 1:

• Evaluated model performance using BLEU score and achieved significant improvements through iterative refinement. Technical Skills
Software Development: Python, OOPS, Django, Flask, FastAPI, SQL, C++, HTML, CSS, Gradio, Streamlit. Artificial Intelligence: Generative AI, Natural Language Processing, Computer Vision, Machine Learning, Deep Learning. Tools and Packages: Hugging Face, LangChain, Ollama, NumPy, Pandas, Scikit-learn, nltk, Pytorch, Keras, Tensorflow. Cloud Services : Azure, AWS
Achievements
• Completed 3 technical internships at small startups, gaining hands-on experience in software development and
problem-solving.
----------------------------------------------------------------------------------------------------
Document 2:

• Trained an CNN model specifically designed to identify and select the optimal frame from a video. Education
Medi-Caps University,Indore 2018 – 2022
B. Tech Information Technology CGPA - 9.2
Paramount Academy, Barwani 2017 – 2018
Higher